# Common Trunk

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# 1. Load the dataset
df = pd.read_csv("/content/uk_weather_historical_2020_2025.csv")
print(df.head())

df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')

# Define Season mapping
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

df['Season'] = df['Month'].apply(get_season)

# Categorical type conversions
df['Location'] = df['Location'].astype('category')
df['Summary'] = df['Summary'].astype('category')

         Date Location  Min_Temp_C  Max_Temp_C  Mean_Temp_C  Humidity_Pct  \
0  2020-01-01   London         2.1         8.0          4.2            85   
1  2020-01-02   London         5.8        13.0          9.6            91   
2  2020-01-03   London         4.6        12.5          9.2            95   
3  2020-01-04   London         6.5        12.3          9.5            89   
4  2020-01-05   London         5.8        14.2          8.8            87   

   Pressure_hPa  Wind_Speed_mph  Cloud_Cover_Pct  Precipitation_mm  \
0        1010.7            13.4               74               0.0   
1        1020.8            10.4               74               0.0   
2        1024.5             3.3               77               5.9   
3        1021.9            13.8               70               0.0   
4        1020.6             9.3               68               0.0   

         Summary  Rain_Tomorrow  
0  Partly Cloudy              0  
1  Partly Cloudy              1  
2          Rai

In [3]:
# 3. Clean the data
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

numeric_cols = ['Min_Temp_C', 'Max_Temp_C', 'Mean_Temp_C', 'Humidity_Pct',
                'Pressure_hPa', 'Wind_Speed_mph', 'Cloud_Cover_Pct', 'Precipitation_mm']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

if 'Summary' in df.columns:
    df['Summary'] = df['Summary'].fillna(df['Summary'].mode()[0])
if 'Location' in df.columns:
    df['Location'] = df['Location'].fillna(df['Location'].mode()[0])

df = df.dropna(subset=['Date']).copy()
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

df['Season'] = df['Month'].apply(get_season)
print('Cleaned shape:', df.shape)
df.head()

Cleaned shape: (2192, 16)


,Date,Location,Min_Temp_C,Max_Temp_C,Mean_Temp_C,Humidity_Pct,Pressure_hPa,Wind_Speed_mph,Cloud_Cover_Pct,Precipitation_mm,Summary,Rain_Tomorrow,Year,Month,Month_Name,Season
0,2020-01-01,London,2.1,8.0,4.2,85,1010.7,13.4,74,0.0,Partly Cloudy,0,2020,1,Jan,Winter
1,2020-01-02,London,5.8,13.0,9.6,91,1020.8,10.4,74,0.0,Partly Cloudy,1,2020,1,Jan,Winter
2,2020-01-03,London,4.6,12.5,9.2,95,1024.5,3.3,77,5.9,Rainy,0,2020,1,Jan,Winter
3,2020-01-04,London,6.5,12.3,9.5,89,1021.9,13.8,70,0.0,Partly Cloudy,0,2020,1,Jan,Winter
4,2020-01-05,London,5.8,14.2,8.8,87,1020.6,9.3,68,0.0,Partly Cloudy,1,2020,1,Jan,Winter


Student B: Rain Prediction System

In [4]:

# 1. Feature Selection & Train/Test Split
features = ['Humidity_Pct', 'Pressure_hPa', 'Wind_Speed_mph', 'Cloud_Cover_Pct']
X = df[features]
y = df['Rain_Tomorrow']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Train Logistic Regression Model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# 3. Evaluation & Metrics
y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 4. Coefficients Extraction and Interpretation
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print("\nModel Coefficients:")
print(coef_df)

Model Accuracy: 0.7188

Confusion Matrix:
[[385  66]
 [119  88]]

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.85      0.81       451
           1       0.57      0.43      0.49       207

    accuracy                           0.72       658
   macro avg       0.67      0.64      0.65       658
weighted avg       0.70      0.72      0.71       658


Model Coefficients:
           Feature  Coefficient
3  Cloud_Cover_Pct     0.118308
2   Wind_Speed_mph     0.018659
1     Pressure_hPa     0.002328
0     Humidity_Pct    -0.135479


Student E: Smart Activity Recommender

In [5]:
# 11. Smart activity recommender functions
activity_rules = {
    'morning_walk': 'Outdoor walk in the early morning',
    'cycling': 'Outdoor cycling activity',
    'football': 'Outdoor sports session',
    'laundry_outside': 'Drying clothes outside'
}

def recommend_activity(rain_prob, temp, wind, activity):
    if activity == 'morning_walk':
        if rain_prob >= 0.7:
            return 'Warning: High chance of rain. Stay home or carry an umbrella.'
        elif temp < 2:
            return 'Too cold for a comfortable walk. Consider indoor exercise.'
        elif wind > 20:
            return 'Windy conditions. Walk only if necessary.'
        else:
            return 'Weather looks suitable for a morning walk.'

    elif activity == 'cycling':
        if rain_prob >= 0.6:
            return 'Cycling is not recommended because rain is likely.'
        elif wind > 18:
            return 'Cycling may be unsafe because of strong wind.'
        else:
            return 'Conditions are acceptable for cycling.'

    elif activity == 'football':
        if rain_prob >= 0.65:
            return 'Outdoor football is not advised. The pitch may be wet.'
        elif temp > 28:
            return 'Too warm for intense outdoor sport. Stay hydrated.'
        else:
            return 'Good weather for outdoor football.'

    elif activity == 'laundry_outside':
        if rain_prob >= 0.4:
            return 'Do not dry clothes outside today. Rain may arrive.'
        elif wind > 22:
            return 'Very windy conditions. Laundry may blow away.'
        else:
            return 'Suitable conditions for drying clothes outside.'

    return 'No rule defined for this activity.'

def predict_and_recommend(input_data):
    input_df = pd.DataFrame([input_data])
    rain_probability = model.predict_proba(input_df)[0, 1]
    temp = input_data['Mean_Temp_C']
    wind = input_data['Wind_Speed_mph']

    print(f'Predicted probability of rain tomorrow: {rain_probability:.2%}\n')

    for activity in activity_rules:
        advice = recommend_activity(rain_probability, temp, wind, activity)
        print(f'{activity}: {advice}')

In [6]:

def predict_and_recommend(input_data):

    input_df_for_prediction = pd.DataFrame([input_data])[features]
    rain_probability = model.predict_proba(input_df_for_prediction)[0, 1]

    temp = input_data['Mean_Temp_C']
    wind = input_data['Wind_Speed_mph']

    print(f'Predicted probability of rain tomorrow: {rain_probability:.2%}\n')

    for activity in activity_rules:
        advice = recommend_activity(rain_probability, temp, wind, activity)
        print(f'{activity}: {advice}')

# Re-run the example recommendation after redefining the function
sample_day = {
    'Min_Temp_C': 11.0,
    'Max_Temp_C': 17.5,
    'Mean_Temp_C': 14.2,
    'Humidity_Pct': 88,
    'Pressure_hPa': 1006.0,
    'Wind_Speed_mph': 30.0,
    'Cloud_Cover_Pct': 82,
    'Precipitation_mm': 3.5
}

predict_and_recommend(sample_day)

Predicted probability of rain tomorrow: 66.39%

morning_walk: Windy conditions. Walk only if necessary.
cycling: Cycling is not recommended because rain is likely.
football: Outdoor football is not advised. The pitch may be wet.
laundry_outside: Do not dry clothes outside today. Rain may arrive.
